# GRU from Scratch: The Simpler Gated Architecture

**What we'll build:** A complete Gated Recurrent Unit (GRU) implemented from scratch in PyTorch, trained on character-level name generation.

**Why it matters:** GRUs provide most of LSTM's benefits (solving vanishing gradients) with fewer parameters and simpler structure. Understanding GRUs reveals how gating mechanisms can be simplified while remaining effective.

**What intuitions we'll develop:**
- How GRUs simplify LSTM's 3 gates into 2 gates
- The role of the **update gate** (combines LSTM's forget + input gates)
- The role of the **reset gate** (controls how much past to forget)
- Why GRUs often match LSTM performance with fewer parameters
- When to choose GRU vs LSTM in practice

## Learning Approach

We'll build understanding incrementally:
1. **Motivation**: Why simplify LSTM?
2. **Architecture**: The two gates of GRU
3. **Math**: Gate equations step-by-step
4. **Implementation**: Build from scratch
5. **Visualization**: See gates in action
6. **Application**: Train on name generation
7. **Comparison**: GRU vs LSTM vs RNN

## 1. Setup

Import necessary libraries and configure the environment.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from aiml_notebooks import (
    create_dataset,
    create_dataloaders,
    get_device,
    set_seed,
)

%load_ext autoreload
%autoreload 2

### Configuration

All hyperparameters in one place for easy experimentation.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed for reproducibility
    
    # Data
    'batch_size': 64,  # Number of samples per training batch
    'train_split': 0.9,  # Fraction of data for training
    
    # Model
    'embedding_dim': 32,  # Size of character embeddings
    'hidden_dim': 128,  # Size of GRU hidden state
    
    # Training
    'learning_rate': 0.003,  # Optimizer learning rate
    'num_epochs': 20,  # Number of training epochs
}

Set random seed for reproducibility and configure device.

In [ ]:
set_seed(CONFIG['seed'])
device = get_device()
print(f"Using device: {device}")

---
## 2. Motivation: Why Simplify LSTM?

### LSTM Recap

LSTMs have **3 gates** and **2 state vectors**:
- **Forget gate** ($f_t$): What to forget from cell state
- **Input gate** ($i_t$): What new information to add
- **Output gate** ($o_t$): What to output from cell state
- **Cell state** ($C_t$): Long-term memory
- **Hidden state** ($h_t$): Short-term memory / output

### The GRU Insight

Researchers at Cho et al. (2014) asked: **Do we need all this complexity?**

GRU simplifies to **2 gates** and **1 state vector**:
- **Update gate** ($z_t$): Combines forget + input gates
- **Reset gate** ($r_t$): Controls access to previous hidden state
- **Hidden state** ($h_t$): Single state that serves both roles

**Benefits:**
- ~25% fewer parameters
- Faster to compute
- Often comparable performance
- Easier to train on smaller datasets

### Visual Comparison

Let's visualize the structural difference between LSTM and GRU.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# LSTM diagram (simplified)
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.set_aspect('equal')
ax1.axis('off')
ax1.set_title('LSTM: 3 Gates, 2 States', fontsize=14, fontweight='bold')

# Gates
for i, (name, color) in enumerate([('Forget', '#FF6B6B'), ('Input', '#4ECDC4'), ('Output', '#45B7D1')]):
    rect = plt.Rectangle((1 + i*2.5, 3), 2, 1.5, facecolor=color, edgecolor='black', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(2 + i*2.5, 3.75, name, ha='center', va='center', fontsize=10, fontweight='bold')

# States
ax1.add_patch(plt.Rectangle((2, 6), 6, 1), )
ax1.text(5, 6.5, 'Cell State (long-term)', ha='center', va='center', fontsize=10)
ax1.add_patch(plt.Rectangle((2, 1), 6, 1))
ax1.text(5, 1.5, 'Hidden State (short-term)', ha='center', va='center', fontsize=10)

# GRU diagram
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_title('GRU: 2 Gates, 1 State', fontsize=14, fontweight='bold')

# Gates
for i, (name, color) in enumerate([('Update', '#9B59B6'), ('Reset', '#F39C12')]):
    rect = plt.Rectangle((1.5 + i*3.5, 3), 2.5, 1.5, facecolor=color, edgecolor='black', linewidth=2)
    ax2.add_patch(rect)
    ax2.text(2.75 + i*3.5, 3.75, name, ha='center', va='center', fontsize=10, fontweight='bold')

# Single state
ax2.add_patch(plt.Rectangle((2, 6), 6, 1.5))
ax2.text(5, 6.75, 'Hidden State (both roles)', ha='center', va='center', fontsize=10)

ax2.text(5, 1, 'No separate cell state!', ha='center', va='center', fontsize=11, 
         style='italic', color='green')

plt.tight_layout()
plt.show()

print("\nLSTM: 3 gates × (W_hh + W_xh + b) = More parameters")
print("GRU:  2 gates × (W_hh + W_xh + b) = ~25% fewer parameters")

---
## 3. GRU Architecture: The Math

### The Two Gates

**Update Gate** ($z_t$): Decides how much of the past to keep vs how much new info to add.
$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t])$$

- When $z_t \approx 1$: Keep previous hidden state (ignore new input)
- When $z_t \approx 0$: Use new candidate state (forget past)

**Reset Gate** ($r_t$): Decides how much of past hidden state to use when computing candidate.
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t])$$

- When $r_t \approx 1$: Use full previous hidden state
- When $r_t \approx 0$: Ignore previous hidden state completely

### Candidate Hidden State

$$\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t])$$

The reset gate $r_t$ controls how much of $h_{t-1}$ flows into the candidate computation.

### Final Hidden State

$$h_t = (1 - z_t) \odot \tilde{h}_t + z_t \odot h_{t-1}$$

This is a **linear interpolation** between old state and candidate:
- $z_t = 0$: $h_t = \tilde{h}_t$ (full update)
- $z_t = 1$: $h_t = h_{t-1}$ (copy-through, like skip connection)

### Intuition: What Each Gate Does

| Gate | High Value (→1) | Low Value (→0) |
|------|----------------|----------------|
| **Update** $z_t$ | Keep old state, ignore new | Fully update to new candidate |
| **Reset** $r_t$ | Use previous hidden state | Start fresh, ignore history |

**Key insight**: The update gate creates a **gradient highway** when $z_t \approx 1$, allowing gradients to flow unchanged through time (similar to ResNet skip connections).

---
## 4. Implementation from Scratch

### GRU Cell

Let's implement a single GRU cell that processes one timestep.

In [ ]:
class GRUCell(nn.Module):
    """
    A single GRU cell implemented from scratch.
    
    Args:
        input_size: Dimension of input features
        hidden_size: Dimension of hidden state
    """
    
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        # Update gate parameters
        self.W_z = nn.Linear(input_size + hidden_size, hidden_size)
        
        # Reset gate parameters
        self.W_r = nn.Linear(input_size + hidden_size, hidden_size)
        
        # Candidate hidden state parameters
        self.W_h = nn.Linear(input_size + hidden_size, hidden_size)
    
    def forward(self, x, h_prev):
        """
        Process one timestep.
        
        Args:
            x: Input at current timestep (batch_size, input_size)
            h_prev: Previous hidden state (batch_size, hidden_size)
        
        Returns:
            h_new: New hidden state (batch_size, hidden_size)
            gates: Dict of gate activations for visualization
        """
        # Concatenate input and previous hidden state
        combined = torch.cat([x, h_prev], dim=1)  # (batch, input_size + hidden_size)
        
        # Update gate: what to keep from past
        z = torch.sigmoid(self.W_z(combined))  # (batch, hidden_size)
        
        # Reset gate: how much past to use for candidate
        r = torch.sigmoid(self.W_r(combined))  # (batch, hidden_size)
        
        # Candidate hidden state (with reset gate applied)
        combined_reset = torch.cat([x, r * h_prev], dim=1)
        h_candidate = torch.tanh(self.W_h(combined_reset))  # (batch, hidden_size)
        
        # Final hidden state: interpolation between old and new
        h_new = (1 - z) * h_candidate + z * h_prev
        
        # Return gates for visualization
        gates = {'update': z, 'reset': r, 'candidate': h_candidate}
        
        return h_new, gates

print("GRUCell implemented!")
print(f"\nGate parameters:")
print(f"  W_z (update gate): input_size + hidden_size → hidden_size")
print(f"  W_r (reset gate):  input_size + hidden_size → hidden_size")
print(f"  W_h (candidate):   input_size + hidden_size → hidden_size")

### Test the GRU Cell

Let's verify our implementation with a simple forward pass.

In [ ]:
# Test GRU cell
input_size = 10
hidden_size = 20
batch_size = 4

cell = GRUCell(input_size, hidden_size)

# Create dummy input and initial hidden state
x = torch.randn(batch_size, input_size)
h_prev = torch.zeros(batch_size, hidden_size)

# Forward pass
h_new, gates = cell(x, h_prev)

print(f"Input shape: {x.shape}")
print(f"Previous hidden: {h_prev.shape}")
print(f"New hidden: {h_new.shape}")
print(f"\nGate shapes:")
print(f"  Update gate: {gates['update'].shape}")
print(f"  Reset gate: {gates['reset'].shape}")
print(f"\nGate statistics (should be ~0.5 for random init):")
print(f"  Update gate mean: {gates['update'].mean():.3f}")
print(f"  Reset gate mean: {gates['reset'].mean():.3f}")

### Full GRU Layer

Now let's create a full GRU layer that processes entire sequences.

In [ ]:
class GRU(nn.Module):
    """
    Full GRU layer that processes sequences.
    
    Args:
        input_size: Dimension of input features
        hidden_size: Dimension of hidden state
    """
    
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = GRUCell(input_size, hidden_size)
    
    def forward(self, x, h_init=None):
        """
        Process a full sequence.
        
        Args:
            x: Input sequence (batch_size, seq_len, input_size)
            h_init: Initial hidden state (batch_size, hidden_size), defaults to zeros
        
        Returns:
            outputs: Hidden states at all timesteps (batch_size, seq_len, hidden_size)
            h_final: Final hidden state (batch_size, hidden_size)
            all_gates: List of gate activations for visualization
        """
        batch_size, seq_len, _ = x.shape
        
        # Initialize hidden state
        if h_init is None:
            h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        else:
            h = h_init
        
        # Process sequence
        outputs = []
        all_gates = []
        
        for t in range(seq_len):
            h, gates = self.cell(x[:, t, :], h)
            outputs.append(h)
            all_gates.append(gates)
        
        # Stack outputs: (batch, seq_len, hidden_size)
        outputs = torch.stack(outputs, dim=1)
        
        return outputs, h, all_gates

print("Full GRU layer implemented!")

---
## 5. Character-Level Language Model

Let's build a complete GRU-based language model for generating names.

In [ ]:
class GRULanguageModel(nn.Module):
    """
    Character-level language model using our GRU.
    """
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = GRU(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        """
        Args:
            x: Input token indices (batch_size, seq_len)
        
        Returns:
            logits: Predictions for next token (batch_size, seq_len, vocab_size)
        """
        # Embed tokens
        embedded = self.embedding(x)  # (batch, seq_len, embedding_dim)
        
        # Pass through GRU
        outputs, _, _ = self.gru(embedded)  # (batch, seq_len, hidden_dim)
        
        # Project to vocabulary
        logits = self.fc(outputs)  # (batch, seq_len, vocab_size)
        
        return logits
    
    def generate(self, start_token, max_len=20, temperature=1.0):
        """
        Generate a name character by character.
        
        Args:
            start_token: Starting token index
            max_len: Maximum generation length
            temperature: Sampling temperature (higher = more random)
        
        Returns:
            generated: List of token indices
        """
        self.eval()
        generated = [start_token]
        h = None
        
        with torch.no_grad():
            for _ in range(max_len):
                # Prepare input
                x = torch.tensor([[generated[-1]]], device=next(self.parameters()).device)
                embedded = self.embedding(x)  # (1, 1, embedding_dim)
                
                # GRU step
                if h is None:
                    outputs, h, _ = self.gru(embedded)
                else:
                    outputs, h, _ = self.gru(embedded, h)
                
                # Get logits and sample
                logits = self.fc(outputs[:, -1, :]) / temperature
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
                
                generated.append(next_token)
                
                # Stop at end token (assuming 0 is end token)
                if next_token == 0:
                    break
        
        return generated

print("GRU Language Model implemented!")

---
## 6. Load Data

Load the names dataset for training our character-level language model.

In [ ]:
full_dataset, train_dataset, val_dataset = create_dataset(
    "names", 
    splits=[CONFIG['train_split'], 1 - CONFIG['train_split']]
)
tokenizer = full_dataset.tokenizer

train_loader, val_loader = create_dataloaders(
    train_dataset, 
    val_dataset, 
    batch_size=CONFIG['batch_size']
)

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Characters: {''.join(tokenizer.chars)}")
print(f"\nDataset sizes:")
print(f"  Training: {len(train_dataset)} names")
print(f"  Validation: {len(val_dataset)} names")
print(f"\nBatch counts:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

Let's examine a sample batch to understand the data format.

In [ ]:
# Get a sample batch
sample_x, sample_y = next(iter(train_loader))

print(f"Input shape: {sample_x.shape}")
print(f"Target shape: {sample_y.shape}")

# Decode an example (filter out -100 padding tokens from target)
example_input = sample_x[0].tolist()
example_target = [t for t in sample_y[0].tolist() if t != -100]

print(f"\nExample name:")
print(f"  Input:  {tokenizer.decode(example_input)}")
print(f"  Target: {tokenizer.decode(example_target)}")
print(f"\nTask: Predict each character given all previous characters")

---
## 7. Training

### Create Model

Instantiate our GRU language model with the configured hyperparameters.

In [ ]:
model = GRULanguageModel(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dim=CONFIG['hidden_dim']
).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {num_params:,} parameters")
print(f"\nArchitecture:")
print(f"  Embedding: {tokenizer.vocab_size} → {CONFIG['embedding_dim']}")
print(f"  GRU: {CONFIG['embedding_dim']} → {CONFIG['hidden_dim']}")
print(f"  Output: {CONFIG['hidden_dim']} → {tokenizer.vocab_size}")

### Training Loop

Define training and evaluation functions.

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)  # (batch, seq_len, vocab_size)
        
        # Reshape for cross entropy
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


@torch.no_grad()
def evaluate(model, val_loader, criterion, device):
    """Evaluate on validation set."""
    model.eval()
    total_loss = 0
    
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss += loss.item()
    
    return total_loss / len(val_loader)

print("Training functions defined!")

### Train the Model

Now let's train our GRU language model!

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': []}

print("Training GRU Language Model...\n")

for epoch in range(CONFIG['num_epochs']):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{CONFIG['num_epochs']} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("\nTraining complete!")

### Training Curves

Visualize how the model learned over time.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], 'b-', linewidth=2, label='Train Loss', marker='o')
plt.plot(history['val_loss'], 'r-', linewidth=2, label='Val Loss', marker='s')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('GRU Training Progress', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"  Train Loss: {history['train_loss'][-1]:.4f}")
print(f"  Val Loss: {history['val_loss'][-1]:.4f}")

---
## 8. Generate Names

Let's see what names our trained GRU can generate!

In [ ]:
print("Generated Names:\n")

for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"Temperature = {temp}:")
    for _ in range(5):
        # Start with the start token (index 0 is '.' which serves as start/end)
        tokens = model.generate(start_token=0, max_len=20, temperature=temp)
        name = tokenizer.decode(tokens[1:-1])  # Remove start/end tokens
        print(f"  {name}")
    print()

---
## 9. Visualize Gate Activations

Let's see how the GRU gates behave when processing a name. This reveals what the model has learned!

In [ ]:
def visualize_gates(model, name, tokenizer, device):
    """Visualize gate activations for a given name."""
    model.eval()
    
    # Encode name
    tokens = tokenizer.encode(name)
    x = torch.tensor([tokens], device=device)
    
    # Get embeddings and run through GRU
    embedded = model.embedding(x)
    _, _, all_gates = model.gru(embedded)
    
    # Extract gate values
    update_gates = torch.stack([g['update'] for g in all_gates]).squeeze().cpu().detach().numpy()
    reset_gates = torch.stack([g['reset'] for g in all_gates]).squeeze().cpu().detach().numpy()
    
    # Average over hidden dimensions for visualization
    update_avg = update_gates.mean(axis=1)
    reset_avg = reset_gates.mean(axis=1)
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    
    chars = [tokenizer.chars[t] for t in tokens]
    x_pos = range(len(chars))
    
    # Update gate
    axes[0].bar(x_pos, update_avg, color='#9B59B6', alpha=0.7)
    axes[0].set_ylabel('Update Gate (avg)', fontsize=11)
    axes[0].set_title(f'Gate Activations for "{name}"', fontsize=14, fontweight='bold')
    axes[0].set_ylim(0, 1)
    axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(chars, fontsize=12)
    
    # Reset gate
    axes[1].bar(x_pos, reset_avg, color='#F39C12', alpha=0.7)
    axes[1].set_ylabel('Reset Gate (avg)', fontsize=11)
    axes[1].set_xlabel('Character', fontsize=11)
    axes[1].set_ylim(0, 1)
    axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(chars, fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    return update_avg, reset_avg

# Visualize for a sample name
_ = visualize_gates(model, "charlotte", tokenizer, device)

print("\nInterpretation:")
print("• High update gate → Keep previous state (memory preservation)")
print("• Low update gate → Update to new state (learn new pattern)")
print("• High reset gate → Use previous hidden state")
print("• Low reset gate → Start fresh, ignore history")

Let's compare gate activations for different names.

In [ ]:
# Compare different names
for name in ["emma", "james", "elizabeth"]:
    _ = visualize_gates(model, name, tokenizer, device)

---
## 10. GRU vs LSTM vs RNN: When to Use What?

### Parameter Comparison

| Architecture | Gates | Parameters (relative) | Speed |
|--------------|-------|----------------------|-------|
| **RNN** | 0 | 1x | Fastest |
| **GRU** | 2 | ~3x | Fast |
| **LSTM** | 3 | ~4x | Slower |

### When to Choose Each

**Use RNN when:**
- Sequences are short (< 10 steps)
- Speed is critical
- Problem is simple

**Use GRU when:**
- You want good long-range modeling with fewer parameters
- Dataset is smaller (fewer params = less overfitting)
- Training speed matters
- Starting a new project (try GRU first!)

**Use LSTM when:**
- You need maximum capacity
- Dataset is large
- Sequences are very long
- Task requires complex gating patterns

**In practice:** GRU and LSTM perform similarly on most tasks. GRU is often the better default due to fewer parameters.

---
## Key Takeaways

1. **GRU simplifies LSTM**: 2 gates instead of 3, single hidden state instead of cell+hidden

2. **Update gate** ($z_t$): Controls the balance between keeping old state and accepting new information
   - $z_t ≈ 1$: Copy previous state (gradient highway)
   - $z_t ≈ 0$: Replace with new candidate

3. **Reset gate** ($r_t$): Controls how much of the past hidden state to use when computing candidate
   - $r_t ≈ 1$: Use full history
   - $r_t ≈ 0$: Ignore history (start fresh)

4. **Gradient flow**: The update gate creates a direct path for gradients, similar to skip connections

5. **Practical choice**: Start with GRU, switch to LSTM only if needed

6. **Modern context**: Both GRU and LSTM are largely superseded by Transformers for most NLP tasks, but remain valuable for:
   - Streaming/online processing
   - Resource-constrained environments
   - Time series with strong temporal dependencies

---
## Exercises

1. **Compare architectures**: Train RNN, GRU, and LSTM on the same data and compare loss curves

2. **Gate analysis**: Which characters trigger high reset gate values? What does this tell you?

3. **Ablation study**: What happens if you remove the reset gate entirely? (Hint: look up "Minimal GRU")

4. **Bidirectional GRU**: Modify the implementation to process sequences in both directions

5. **Multi-layer GRU**: Stack multiple GRU layers and compare performance